# 🛰️ main.ipynb — Mapping Natural-Disaster Damage from Satellite Imagery

**Team:** Bhuvanesh Dinesh Wadhwani · Artur Zavistovskyi · Anastasiia Khitrova · Yitian Zhou

This is **the** notebook for the project. It runs the whole story end-to-end:
**Part 1** (data + EDA) → **Part 2** (two models + held-out evaluation) → **Part 3** (impact analysis).

### ▶️ One switch controls everything: `USE_SYNTHETIC`
- `USE_SYNTHETIC = True` → built-in **quick test**: generates toy data, runs the *entire* pipeline in
  ~3–6 min on CPU. **No download needed.** Use this to see how everything works / to develop.
- `USE_SYNTHETIC = False` → real **xBD / xView2** data (download first, see `READ_EVERYBODY.md`).
  Same models, same code — only the data source changes.

> The heavy code lives in `src/` (`config`, `data_utils`, `models`, `train_utils`) so this notebook
> stays readable and the team can edit `.py` files without notebook merge conflicts.

> ⚠️ With synthetic data the accuracy numbers are toy numbers — they only prove the machinery works.
> Real conclusions come from the xBD run.

## 0 · Setup & the master switch

In [ ]:
import sys, os, time
sys.path.append(os.path.abspath('.'))     # so 'from src import ...' works
import numpy as np, matplotlib.pyplot as plt, torch
from torch.utils.data import DataLoader
from src import config, data_utils, models, train_utils

# ⬇️⬇️⬇️  THE ONE SWITCH  ⬇️⬇️⬇️
USE_SYNTHETIC = True      # True = quick test on toy data (no download). False = real xBD.
# ⬆️⬆️⬆️                  ⬆️⬆️⬆️

torch.manual_seed(config.SEED); np.random.seed(config.SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Mode   :', 'SYNTHETIC (quick test)' if USE_SYNTHETIC else 'REAL xBD')
print('Device :', device)
print('Train on:', config.TRAIN_DISASTERS, '| Hold out for test:', config.HELDOUT_DISASTER)

# Part 1 · Data Collection & Preliminary Analysis  *(10 pts)*

We classify damage **per building** by comparing a `pre` and `post` patch. The held-out disaster
(`wildfire`) is kept entirely for testing — that's the cross-disaster transfer question.

### 🔍 Data validation — what's actually in the dataset?
Before modeling, let's sanity-check the **raw xView2 tiles**: which disaster types we have, and
how many buildings fall in each damage class. (Runs only on real data; skipped in synthetic mode.)

In [ ]:
import json, collections
from pathlib import Path
sources = data_utils.find_xbd_sources()
print('xView2 source folders:', [str(s) for s in sources] or 'NONE — using synthetic data')

if sources:
    tiles_by_disaster = collections.Counter()
    buildings_by_class = collections.Counter()
    for root in sources:
        for jp in (root / 'labels').glob('*post_disaster.json'):
            dtype = config.event_to_disaster_type(jp.stem.split('_')[0])
            tiles_by_disaster[dtype] += 1
            feats = json.loads(jp.read_text())['features'].get('xy', [])
            for f in feats:
                buildings_by_class[f['properties'].get('subtype', 'un-classified')] += 1
    print(f"\nScanned {sum(tiles_by_disaster.values())} post-disaster tiles")
    print('\nTiles per disaster type (held-out =', config.HELDOUT_DISASTER, '):')
    for k, v in tiles_by_disaster.most_common():
        flag = '  <- HELD-OUT (test)' if k == config.HELDOUT_DISASTER else ''
        print(f'  {k:12s} {v:5d}{flag}')
    print('\nBuildings per damage class:')
    total = sum(buildings_by_class.values())
    for k in config.DAMAGE_CLASSES + ['un-classified']:
        v = buildings_by_class.get(k, 0)
        print(f'  {k:14s} {v:7d}  ({100*v/max(total,1):4.1f}%)')
else:
    print('(Extract the xView2 .tar into data/train and set USE_SYNTHETIC=False to validate real data.)')

In [ ]:
# Peek at one real PRE/POST tile per disaster type, so you SEE what the model gets.
if sources:
    from PIL import Image
    samples = {}
    for root in sources:
        for jp in sorted((root / 'labels').glob('*post_disaster.json')):
            dtype = config.event_to_disaster_type(jp.stem.split('_')[0])
            if dtype not in samples:
                pre = root / 'images' / (jp.stem.replace('post', 'pre') + '.png')
                post = root / 'images' / (jp.stem + '.png')
                if pre.exists() and post.exists():
                    samples[dtype] = (pre, post)
        if len(samples) >= 4:
            break
    n = len(samples)
    fig, axes = plt.subplots(n, 2, figsize=(7, 3.4 * n))
    axes = axes.reshape(n, 2)
    for r, (dtype, (pre, post)) in enumerate(samples.items()):
        axes[r, 0].imshow(Image.open(pre));  axes[r, 0].set_title(f'{dtype} · PRE')
        axes[r, 1].imshow(Image.open(post)); axes[r, 1].set_title(f'{dtype} · POST')
    for ax in axes.ravel(): ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('Skipped (synthetic mode).')

### Load the dataset (build building-patches on first real run)

In [ ]:
if USE_SYNTHETIC:
    # toy 'buildings before/after' (see src/data_utils.make_pair)
    train_ds, val_ds, test_ds = data_utils.make_synthetic_datasets()
else:
    # First real run crops building patches from the tiles -> data/processed/manifest.csv.
    # limit_tiles keeps this FAST for a first pass (covers all disaster types via shuffle).
    # Set limit_tiles=None below once you want the FULL dataset (slower, many patches).
    manifest = os.path.join(config.DATA_DIR, 'processed', 'manifest.csv')
    if not os.path.exists(manifest):
        data_utils.build_xbd_manifest(limit_tiles=500)
    train_ds = data_utils.DamageDataset('train', augment=True)
    val_ds   = data_utils.DamageDataset('val',   augment=False)
    test_ds  = data_utils.DamageDataset('test',  augment=False)

print(f'train {len(train_ds)} | val {len(val_ds)} | test/held-out {len(test_ds)}')
print('Class counts in train:', np.bincount(train_ds.labels, minlength=4),
      '=', dict(zip(config.DAMAGE_CLASSES, np.bincount(train_ds.labels, minlength=4))))

### Class distribution ⚖️
Most buildings are undamaged — this imbalance is why we use class weights later.

In [ ]:
counts = np.bincount(train_ds.labels, minlength=4)
plt.figure(figsize=(6, 3))
plt.bar(config.DAMAGE_CLASSES, counts, color=['#2ca02c', '#ffdf00', '#ff7f0e', '#d62728'])
plt.title('Damage class distribution (train)'); plt.ylabel('patches'); plt.xticks(rotation=20)
plt.tight_layout(); plt.show()
# For real xBD you can also do per-disaster: pd.crosstab(df.disaster_type, df.label)

### Example pre/post patches 🖼️
Top row = before, bottom row = after. Notice damage grows left→right.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
for col, d in enumerate(range(4)):
    idx = np.where(train_ds.labels == d)[0]
    if len(idx) == 0:
        continue
    pre, post = train_ds.raw(idx[0])
    axes[0, col].imshow(data_utils.denormalize(pre));  axes[0, col].set_title(f'PRE · {config.DAMAGE_CLASSES[d]}')
    axes[1, col].imshow(data_utils.denormalize(post)); axes[1, col].set_title(f'POST · {config.DAMAGE_CLASSES[d]}')
for ax in axes.ravel(): ax.axis('off')
plt.tight_layout(); plt.show()

# Part 2 · Deep Learning Models  *(10 pts)*

Two **siamese (two-branch, shared-weight)** change-detection CNNs, compared fairly:
1. **from scratch** — a compact custom CNN
2. **pretrained** — a fine-tuned ResNet18 backbone

Both combine the two branch features as `[pre, post, |pre−post|]` (the abs-difference is the *change*
signal) and predict the 4 damage classes. Code: `src/models.py`.

In [ ]:
train_dl = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=config.BATCH_SIZE)
test_dl  = DataLoader(test_ds,  batch_size=config.BATCH_SIZE)

# class weights so the model can't cheat by always predicting 'no-damage'
counts = np.bincount(train_ds.labels, minlength=4)
class_weights = torch.tensor(counts.sum() / (4 * np.maximum(counts, 1)), dtype=torch.float32)
print('class weights:', class_weights.numpy().round(2))

### Train the from-scratch CNN 🧱

In [ ]:
t0 = time.time()
scratch = models.SiameseScratchCNN(config.NUM_CLASSES)
train_utils.train_model(scratch, train_dl, val_dl, device, epochs=config.EPOCHS,
                        class_weights=class_weights, ckpt_path='checkpoints/scratch.pt')
print(f'⏱️ scratch trained in {time.time()-t0:.1f}s')

### Train the pretrained ResNet18 🏋️
First run downloads ImageNet weights (~45 MB). No internet? It falls back to random init automatically.

In [ ]:
t0 = time.time()
pretrained = models.SiamesePretrainedCNN(config.NUM_CLASSES, backbone='resnet18', freeze_until=6)
train_utils.train_model(pretrained, train_dl, val_dl, device, epochs=config.EPOCHS,
                        class_weights=class_weights, ckpt_path='checkpoints/pretrained.pt')
print(f'⏱️ pretrained trained in {time.time()-t0:.1f}s')

## 🎯 The key result: evaluate on a disaster nobody trained on
We compare both models on **SEEN** disasters (val) vs the **HELD-OUT** disaster (test).
A big drop on held-out = the transfer gap — the whole point of the project.

In [ ]:
import pandas as pd
rows = []; cms = {}
for name, model in [('scratch', scratch), ('pretrained', pretrained)]:
    for split, dl in [('SEEN (val)', val_dl), ('HELD-OUT (test)', test_dl)]:
        y, p = train_utils.predict(model, dl, device)
        cm, acc, f1, _ = train_utils.scores(y, p, config.NUM_CLASSES)
        cms[(name, split)] = cm
        rows.append({'model': name, 'split': split, 'accuracy': round(acc, 3), 'macro_F1': round(f1, 3)})
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
comparison

In [ ]:
# Confusion matrices (rows=true, cols=pred). Diagonal = correct.
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, ((name, split), cm) in zip(axes.ravel(), cms.items()):
    train_utils.plot_confusion(cm, config.DAMAGE_CLASSES, ax=ax, title=f'{name} · {split}')
plt.tight_layout(); plt.show()

## 🗺️ Deployment: a damage map
Predict every building in an affected area and color it green→red by damage level — the picture
responders would actually use.

In [ ]:
if USE_SYNTHETIC:
    GRID = 6
    rng = np.random.default_rng(7)
    pres, posts, truth = [], [], []
    for _ in range(GRID*GRID):
        d = int(rng.choice(4, p=data_utils.CLASS_P))
        a, b = data_utils.make_pair(d, config.HELDOUT_DISASTER, rng)   # area hit by held-out disaster
        pres.append(a); posts.append(b); truth.append(d)
    Xpre  = torch.stack([data_utils.normalize_chw(a) for a in pres])
    Xpost = torch.stack([data_utils.normalize_chw(b) for b in posts])
    with torch.no_grad():
        pred = pretrained(Xpre.to(device), Xpost.to(device)).argmax(1).cpu().numpy()
    COLOR = np.array([[.1,.7,.1],[.95,.85,.1],[.95,.5,.1],[.85,.1,.1]])
    P = config.PATCH_SIZE; mosaic = np.ones((GRID*P, GRID*P, 3), np.float32)
    for k, b in enumerate(posts):
        r, c = divmod(k, GRID); mosaic[r*P:(r+1)*P, c*P:(c+1)*P] = b
    fig, ax = plt.subplots(1, 3, figsize=(15, 5.5))
    ax[0].imshow(mosaic); ax[0].set_title('POST imagery (area)')
    ax[1].imshow(COLOR[np.array(truth).reshape(GRID, GRID)]); ax[1].set_title('True damage')
    ax[2].imshow(COLOR[pred.reshape(GRID, GRID)]); ax[2].set_title('Predicted damage')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()
    print('map accuracy:', round((pred == np.array(truth)).mean(), 2),
          '| green=ok, yellow=minor, orange=major, red=destroyed')
else:
    print('TODO(real data): pick a held-out xBD event tile, crop all its building patches,')
    print('predict with the better model, and draw each footprint colored by predicted class.')

# Part 3 · Impact & Recovery Analysis  *(10 pts)*  — *open for the team* ✍️

This part is **freeform** — grab it and run. The goal: overlay our predicted damage with
socioeconomic data and ask whether damage/recovery is **unequally distributed** across communities.

Steps to fill in:
1. Get socioeconomic data (income / population density / housing) for an affected region
   — e.g. US Census ACS for a US hurricane. (Use the xBD geotransforms for lat/lon.)
2. Aggregate predicted damage per community (census tract / grid cell).
3. Correlate damage rate (and recovery, if multi-date) with income & density. Plot it + a map.
4. Write up: how rapid maps help responders, the risk of acting on *predicted* vs *verified* damage,
   limitations (label noise, cloud/smoke, imagery availability), and the ethics.

> 💡 Nothing here can break Parts 1–2 — this section only *reads* the model's predictions.

In [ ]:
# TODO(team): load socioeconomic table + predicted-damage-by-location, join, analyze, plot.
# import geopandas as gpd   # handy for spatial joins + choropleth maps
print('Part 3 is yours — see the steps above. Ping the chat if you want a starting scaffold.')

## ✅ Done
You ran the full project: data → EDA → two models → held-out evaluation → damage map → (impact).

**To go from test to real:** download xBD (see `READ_EVERYBODY.md`), set `USE_SYNTHETIC = False`,
and re-run. Everything else stays the same.